In [1]:
import sys, os
sys.path.append(os.path.dirname('__file__'))
sys.path.append(os.getcwd())
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
import pickle 
import numpy as np
sys.path.append(os.path.dirname(os.getcwd()))
from pyfrechet.metrics import mse
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from joblib import Parallel, delayed

from pyfrechet.metric_spaces import MetricData, LogEuclidean
from pyfrechet.metric_spaces import MetricData, LogEuclidean
from pyfrechet.regression.bagged_regressor import BaggedRegressor
from pyfrechet.regression.trees import Tree
from pyfrechet.metric_spaces.utils import vectorize, devectorize


# By-blocks execution
n_samples = len(os.listdir(os.path.join(os.getcwd(), 'simulations_SPD/data')))
n_cores = 4  # Adjusted for your MacBook Air (4 physical cores)
n_blocks = int(np.ceil(n_samples / n_cores))
current_block = 1

# Define parameter grid for tuning
param_grid = {
    'estimator__min_split_size': [1]
}

# Custom scorer (negative mean squared error)
neg_mse = make_scorer(mse, greater_is_better=False)

def tune_forest(X, y):
    """ Perform hyperparameter tuning using GridSearchCV. """
    base = Tree(split_type='2means', mtry=None, impurity_method='cart')
    forest = BaggedRegressor(estimator=base, n_estimators=3, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)
    
    tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=5, n_jobs=1, verbose=4)
    tuned_forest.fit(X, y)
    return tuned_forest.best_estimator_

with open(os.path.join(os.getcwd(), 'simulations_SPD/data/' + 'SPD_Samp2_N100_df5.pkl'), 'rb') as f:
    sample = pickle.load(f)

X = np.c_[sample['t']]
sample_Y = np.array(sample['y'])

for dist in ['AI']:
    if dist == 'AI':
        M = LogEuclidean(dim=2)
        # To do: turn SPDVectorizer into a function, it is not necessary to define a class
        y_vectors = vectorize(sample_Y)
        y = MetricData(M, y_vectors)

        # Define the base tree and forest
        base = Tree(split_type='2means', impurity_method='cart', mtry=None, min_split_size=1)
        forest = BaggedRegressor(estimator=base, n_estimators=3, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)

        # Perform hyperparameter tuning
        param_grid = {'estimator__min_split_size': [1, 5]}
        neg_mse = make_scorer(mse, greater_is_better=False)

        tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=5, n_jobs=1, verbose=4)
        tuned_forest.fit(X, y)  # Use vectorized data

        best_forest = tuned_forest.best_estimator_
        best_forest.fit(X, y)  # Refit with best params

        results = {
            'x_train_data': X,
            'y_train_data': y_vectors,
            'train_predictions': best_forest.predict(X),
            'forest': best_forest,
        }
results
print(f'Block number: {current_block}')

INFO: Using numpy backend


Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV 1/5] END ......estimator__min_split_size=1;, score=-2.976 total time=   0.3s
[CV 2/5] END ......estimator__min_split_size=1;, score=-2.510 total time=   0.3s
[CV 3/5] END ......estimator__min_split_size=1;, score=-2.057 total time=   0.3s
[CV 4/5] END ......estimator__min_split_size=1;, score=-2.933 total time=   0.3s
[CV 5/5] END ......estimator__min_split_size=1;, score=-1.850 total time=   0.3s
[CV 1/5] END ......estimator__min_split_size=5;, score=-3.394 total time=   0.2s
[CV 2/5] END ......estimator__min_split_size=5;, score=-1.829 total time=   0.1s
[CV 3/5] END ......estimator__min_split_size=5;, score=-2.101 total time=   0.1s
[CV 4/5] END ......estimator__min_split_size=5;, score=-2.763 total time=   0.1s
[CV 5/5] END ......estimator__min_split_size=5;, score=-1.636 total time=   0.1s
Block number: 1
